# Encoder output inspection — nominal vs hard-flip (frozen nominal checkpoint)

Loads the **frozen nominal** transformer checkpoint (`transformer_jet_classifier_nominal.pt` — predates the auto-scaler work, plain architecture, no `flip_scale`) and compares its encoder output (the CLS embedding, `h[:, 0]`) for nominal vs the **original hard flip**: sign-negate the 4 impact-parameter fields (`d0`, `z0SinTheta`, `lifetimeSignedD0Significance`, `lifetimeSignedZ0SinThetaSignificance`) on the *same* selected top-k tracks (no track re-ranking, consistent with the auto-scaler work's convention). This is the physics-prior flip itself, not a learned one — no auto-scaler / `flip_scale` involved here.

Workflow:
1. Run the **Config**, **Data loading & model definition**, and **Load model + data** / **Build hard-flip view & run inference** cells once.
2. Re-run any plotting cell as many times as you like.

In [ ]:
import glob
import hashlib
import json
import os

import h5py
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde


## Config

`CKPT_DIR` points at the nominal run's output directory — architecture, track fields, and the 4 hard-flip fields are read from its `config.json`. `CACHE_DIR` points at an existing cache of pre-computed nominal + hard-flip (origins 3 & 4) track arrays; no H5 file is read and no indices are (re-)derived — see the next section.

In [ ]:
CKPT_DIR = "/Users/adminbingxuanliu/Work/SALTOpenData/FlipStudies/results/transformer_results_nominal/"
DATA_FILE = "mc-flavtag-ttbar-small.h5"       # unused now — kept only for the dormant load_tracks() fallback below
CACHE_DIR = "../FlipStudies/results/val_cache/"  # pre-existing nominal + hard-flip (origins 3,4) cache
N_TEST = 1_200_000
N_PLOT_SAMPLE = 1_200_000  # subsample size for scatter/PCA plots (histograms use the full set)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(os.path.join(CKPT_DIR, "config.json")) as _f:
    run_cfg = json.load(_f)

TOP_K            = run_cfg["top_k"]
TRACK_FIELDS     = run_cfg["track_fields"]
FLIP_FIELDS      = run_cfg["flip_fields"]  # the 4 physics-motivated fields to hard-negate
# None = flip every track; a list of origin ints = only flip tracks with that true origin.
# Defaults to this run's own (previously unused) config value.
# 0=Pileup 1=Fake 2=Primary 3=From b 4=From b->c 5=From c 6=From tau 7=Other secondary
FLIP_ORIGINS     = run_cfg.get("flip_origins")
#FLIP_ORIGINS = [0,1,2,3,4,5,6,7]
ORIGIN_NAMES     = ["Pileup", "Fake", "Primary", "From b", "From b->c", "From c", "From tau", "Other sec."]
FLAVOUR_TO_LABEL = {int(k): v for k, v in run_cfg["flavour_to_label"].items()}
CLASS_NAMES      = run_cfg["class_names"]
COLOURS          = run_cfg["colours"]        
D_MODEL          = run_cfg["d_model"]
N_HEADS          = run_cfg["n_heads"]
N_LAYERS         = run_cfg["n_layers"]
D_FFN            = run_cfg["d_ffn"]
DROPOUT          = run_cfg["dropout"]
N_ORIGINS        = run_cfg["n_origins"]
N_FEATS          = len(TRACK_FIELDS)
FLIP_COL_IDX     = [TRACK_FIELDS.index(f) for f in FLIP_FIELDS]
CHECKPOINT_PATH  = os.path.join(CKPT_DIR, run_cfg["model_name"])

os.makedirs(CACHE_DIR, exist_ok=True)
PLOT_DIR = os.path.join(CKPT_DIR, "encoder_inspection_hardflip/")
os.makedirs(PLOT_DIR, exist_ok=True)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Hard-flip fields: {FLIP_FIELDS}")

if FLIP_ORIGINS is None:
    print("Flipping ALL tracks regardless of origin")
else:
    print(f"Flipping only tracks with origin in {FLIP_ORIGINS} ({[ORIGIN_NAMES[o] for o in FLIP_ORIGINS]})")

## Data loading & model definition

In [ ]:
def _cache_key(idx):
    h = hashlib.md5(idx.tobytes()).hexdigest()[:12]
    return os.path.join(CACHE_DIR, f"tracks_{h}_nom.npz")


def load_tracks(path, idx):
    """Returns (N, K, F) features, (N, K) validity mask, (N,) labels, (N, K) origins."""
    cp = _cache_key(idx)
    if os.path.exists(cp):
        d = np.load(cp)
        if "origins" in d:
            return d["X"], d["mask"], d["y"], d["origins"]

    with h5py.File(path, "r") as f:
        flavour_id = f["jets"]["HadronConeExclTruthLabelID"][idx]
        keep_jet   = np.isin(flavour_id, list(FLAVOUR_TO_LABEL.keys()))
        fidx       = idx[keep_jet]

        valid  = f["tracks"]["valid"][fidx]
        d0     = f["tracks"]["d0"][fidx].astype(np.float32)
        ip2d   = f["tracks"]["lifetimeSignedD0Significance"][fidx].astype(np.float32)
        origin = f["tracks"]["GN2v01_trackOrigin"][fidx].astype(np.int8)
        arrs   = {fld: f["tracks"][fld][fidx].astype(np.float32) for fld in TRACK_FIELDS}

    keep = valid & (np.abs(d0) < 3.5)

    sort_key = ip2d.copy()
    sort_key[~keep] = -np.inf
    order = np.argsort(-sort_key, axis=1)

    feat_list = [arrs[fld] for fld in TRACK_FIELDS]
    feats = np.stack(feat_list, axis=-1)

    topk_idx    = order[:, :TOP_K]
    rows        = np.arange(len(fidx))[:, None]
    topk_feat   = feats[rows, topk_idx]
    topk_valid  = keep[rows, topk_idx]
    topk_feat   = np.where(topk_valid[:, :, None], topk_feat, 0.0).astype(np.float32)
    topk_origin = origin[rows, topk_idx].astype(np.int64)
    topk_origin[~topk_valid] = -1

    labels = np.array([FLAVOUR_TO_LABEL[v] for v in flavour_id[keep_jet]], dtype=np.int64)

    np.savez(cp, X=topk_feat, mask=topk_valid, y=labels, origins=topk_origin)
    return topk_feat, topk_valid, labels, topk_origin

In [ ]:
# Plain JetTransformer — matches the frozen nominal checkpoint's actual architecture:
# no flip_scale/flip_mask, no `flip` argument. This predates the auto-scaler work.
class JetTransformer(nn.Module):
    def __init__(self, in_dim, d_model, n_heads, n_layers, d_ffn, dropout,
                 n_classes, n_origins):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, d_model)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ffn,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder     = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier  = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, n_classes),
        )
        self.origin_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, n_origins),
        )

    def forward(self, x, mask):
        B = x.size(0)
        h = self.input_proj(x)

        cls = self.cls_token.expand(B, -1, -1)
        h   = torch.cat([cls, h], dim=1)

        cls_valid            = torch.ones(B, 1, dtype=torch.bool, device=x.device)
        src_key_padding_mask = ~torch.cat([cls_valid, mask], dim=1)

        h = self.encoder(h, src_key_padding_mask=src_key_padding_mask)
        # h[:, 0]  -> CLS token  -> jet classification / encoder output we're inspecting
        # h[:, 1:] -> track tokens -> per-track origin classification
        return self.classifier(h[:, 0]), self.origin_head(h[:, 1:]), h[:, 0]

## Load model + data

In [ ]:
model = JetTransformer(N_FEATS, D_MODEL, N_HEADS, N_LAYERS, D_FFN, DROPOUT,
                       n_classes=3, n_origins=N_ORIGINS).to(DEVICE)
state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
missing, unexpected = model.load_state_dict(state, strict=True)
model.eval()
print(f"Loaded {CHECKPOINT_PATH} (strict=True, no missing/unexpected keys)")

In [ ]:
# Load directly from the pre-existing, already-computed cache (no H5 read, no RNG) —
# a matched nominal/flip34 pair, e.g. from validate_transformer_pred_flip.ipynb's own run.
# '_flip34' means it was built with the hard flip restricted to origins 3 & 4 (From b,
# From b->c) — the same FLIP_ORIGINS default read from this checkpoint's config above.
nom_files = sorted(glob.glob(os.path.join(CACHE_DIR, "tracks_*_nom.npz")))
assert len(nom_files) == 1, f"Expected exactly one *_nom.npz in {CACHE_DIR}, found: {nom_files}"
nom_file  = nom_files[0]
cache_tag = os.path.basename(nom_file)[len("tracks_"):-len("_nom.npz")]
flip_file = os.path.join(CACHE_DIR, f"tracks_{cache_tag}_flip34.npz")
assert os.path.exists(flip_file), f"No matching flip cache for {nom_file}: expected {flip_file}"

print(f"Loading nominal cache: {nom_file}")
_d_nom  = np.load(nom_file, mmap_mode="r")
print(f"Loading flip cache:    {flip_file}")
_d_flip = np.load(flip_file, mmap_mode="r")

n_total = len(_d_nom["y"])
n_use   = min(N_TEST, n_total) if N_TEST is not None else n_total

# mmap_mode='r' + slicing before materialising keeps this from pulling the full
# multi-GB arrays into memory when n_use < n_total.
X_test       = np.array(_d_nom["X"][:n_use])
mask_test    = np.array(_d_nom["mask"][:n_use])
y_test       = np.array(_d_nom["y"][:n_use])
origins_test = np.array(_d_nom["origins"][:n_use])
X_flip       = np.array(_d_flip["X"][:n_use])

assert np.array_equal(_d_flip["y"][:n_use], y_test), "nominal/flip caches have different jets"
assert np.array_equal(_d_flip["mask"][:n_use], mask_test), "nominal/flip caches have different valid-track masks"

print(f"Using {n_use:,} / {n_total:,} cached jets — "
      f"b:{(y_test==0).sum():,}  c:{(y_test==1).sum():,}  light:{(y_test==2).sum():,}")

## Run inference (nominal vs pre-computed hard-flip)

`X_flip` was already loaded above straight from the matching `_flip34` cache file — no on-the-fly construction here. Just run the frozen model over both.

In [ ]:
X_nom_t  = torch.from_numpy(X_test)
X_flip_t = torch.from_numpy(X_flip)
mask_t   = torch.from_numpy(mask_test)

BATCH = 1024
h_nom_list, h_flip_list, p_nom_list, p_flip_list = [], [], [], []
with torch.no_grad():
    for i in range(0, len(y_test), BATCH):
        mask_b = mask_t[i:i+BATCH].to(DEVICE)
        logits_nom,  _, h_nom_b  = model(X_nom_t[i:i+BATCH].to(DEVICE),  mask_b)
        logits_flip, _, h_flip_b = model(X_flip_t[i:i+BATCH].to(DEVICE), mask_b)
        h_nom_list.append(h_nom_b.cpu())
        h_flip_list.append(h_flip_b.cpu())
        p_nom_list.append(torch.softmax(logits_nom, dim=1).cpu())
        p_flip_list.append(torch.softmax(logits_flip, dim=1).cpu())

h_nom  = torch.cat(h_nom_list).numpy()   # (N, d_model)
h_flip = torch.cat(h_flip_list).numpy()  # (N, d_model)
p_nom  = torch.cat(p_nom_list).numpy()   # (N, 3)
p_flip = torch.cat(p_flip_list).numpy()  # (N, 3)
y      = y_test
print(f"h_nom {h_nom.shape}, h_flip {h_flip.shape}")

## Plot: cosine similarity of encoder output, nominal vs hard-flip

Light should cluster near 1 (invariant to the hard flip), b should be pulled away from 1.

In [ ]:
cos_sim = np.sum(h_nom * h_flip, axis=1) / (
    np.linalg.norm(h_nom, axis=1) * np.linalg.norm(h_flip, axis=1) + 1e-10)

FLAVOUR_STYLES = {'b-jet': "-", 'c-jet': "--",'light-jet': "-."}
fig, ax = plt.subplots(figsize=(7, 7))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = y == cls_idx
    ax.hist(cos_sim[m], bins=50, range=(0.25, 1), histtype="step",
            label=cls_name, color=COLOURS[cls_name], linewidth=2, linestyle=FLAVOUR_STYLES[cls_name],density=True)
ax.set_yscale("log")
ax.set_xlabel("Cosine Similarity", fontsize=12, loc="right"); 
ax.set_ylabel("Density", fontsize=12, loc="top")
#ax.set_title("Encoder output similarity, nominal vs hard-flip")
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(PLOT_DIR + "cos_sim_by_flavour.png", dpi=150, bbox_inches="tight")
print("Saved cos_sim_by_flavour.png")
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = y == cls_idx
    print(f"  {cls_name:10s} mean cos_sim = {cos_sim[m].mean():+.4f}")

## Plot: PCA of encoder output — nominal vs hard-flip, overlaid 90% contours

Fits PCA with 3 components jointly on nominal + hard-flip embeddings (shared basis), then for each pair of components (PC1-PC2, PC2-PC3, PC1-PC3) draws the 90%-density contour of each flavour's nominal (solid) and hard-flip (dashed) distribution, overlaid on one canvas — instead of separate nominal/flip scatter panels.

In [ ]:
rng_plot = np.random.default_rng(0)
sel = rng_plot.choice(len(y), size=min(N_PLOT_SAMPLE, len(y)), replace=False)

pca = PCA(n_components=3)
pca.fit(np.concatenate([h_nom, h_flip], axis=0))
nom_3d  = pca.transform(h_nom[sel])   # (N_sel, 3)
flip_3d = pca.transform(h_flip[sel])  # (N_sel, 3)
y_sel   = y[sel]


def contour_grid(x, y, grid_n=150):
    """KDE-estimated density on a grid, plus the density threshold enclosing `level`
    of the total probability mass (used to draw a single 90%-highest-density contour)."""
    kde = gaussian_kde(np.vstack([x, y]))
    gx = np.linspace(x.min(), x.max(), grid_n)
    gy = np.linspace(y.min(), y.max(), grid_n)
    XX, YY = np.meshgrid(gx, gy)
    Z = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(grid_n, grid_n)
    return XX, YY, Z


def density_threshold(Z, level=0.95):
    z_flat  = np.sort(Z.ravel())[::-1]
    cumsum  = np.cumsum(z_flat) / z_flat.sum()
    return z_flat[np.searchsorted(cumsum, level)]


PC_PAIRS = [(0, 1), (1, 2), (0, 2)]
CONDITION_STYLES = {"nominal": "-", "all-flip": "--"}
LOCS = ["upper right", "lower right","lower right"]

i = 0
for pi, pj in PC_PAIRS:
    fig, ax = plt.subplots(figsize=(7, 7))
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        m = y_sel == cls_idx
        for arr, cond in [(nom_3d, "nominal"), (flip_3d, "all-flip")]:
            x_, y_ = arr[m, pi], arr[m, pj]
            XX, YY, Z = contour_grid(x_, y_)
            thr = density_threshold(Z)
            ax.contour(XX, YY, Z, levels=[thr], colors=[COLOURS[cls_name]],
                       linestyles=CONDITION_STYLES[cond], linewidths=3)
            ax.plot([], [], color=COLOURS[cls_name], linestyle=CONDITION_STYLES[cond],
                    label=f"{cls_name} ({cond})")
    ax.set_xlabel(f"PC{pi+1}", loc="right", fontsize=12); ax.set_ylabel(f"PC{pj+1}", loc="top", fontsize=12)
    #ax.set_title(f"Encoder output 90% contours — PC{pi+1} vs PC{pj+1}")
    ax.legend(fontsize=12, loc=LOCS[i])
    i = i + 1
    plt.tight_layout()
    fname = f"pca_contours_PC{pi+1}_vs_PC{pj+1}.png"
    plt.savefig(PLOT_DIR + fname, dpi=150, bbox_inches="tight")
    print(f"Saved {fname}")

In [ ]:
# Movement lines for one flavour at a time — set CLASS_TO_TRACE below. Uses PC1/PC2 of
# the 3-component fit above.
CLASS_TO_TRACE = "b-jet"
trace_idx = CLASS_NAMES.index(CLASS_TO_TRACE)
m = y_sel == trace_idx
n_lines = min(300, m.sum())
line_sel = rng_plot.choice(np.where(m)[0], size=n_lines, replace=False)

fig, ax = plt.subplots(figsize=(7, 7))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mm = y_sel == cls_idx
    ax.scatter(nom_3d[mm, 0], nom_3d[mm, 1], s=4, alpha=0.15,
               color=COLOURS[cls_name], label=f"{cls_name} (nom)", linewidths=0)
for i in line_sel:
    ax.plot([nom_3d[i, 0], flip_3d[i, 0]], [nom_3d[i, 1], flip_3d[i, 1]],
            color=COLOURS[CLASS_TO_TRACE], linewidth=0.6, alpha=0.6)
ax.scatter(flip_3d[line_sel, 0], flip_3d[line_sel, 1], s=10, marker="x",
           color=COLOURS[CLASS_TO_TRACE], label=f"{CLASS_TO_TRACE} (hard-flip)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(fontsize=8)
ax.set_title(f"Nom -> hard-flip movement for a sample of {CLASS_TO_TRACE} jets")
plt.tight_layout()
plt.savefig(PLOT_DIR + f"pca_movement_{CLASS_TO_TRACE.replace('-', '_')}.png", dpi=150, bbox_inches="tight")
print(f"Saved pca_movement_{CLASS_TO_TRACE.replace('-', '_')}.png")

## Plot: per-dimension embedding sensitivity

Which of the `d_model` latent dimensions does the hard flip perturb most, on average, for each flavour?

In [ ]:
delta = h_flip - h_nom  # (N, d_model)
fig, ax = plt.subplots(figsize=(9, 5))
width = 0.8 / len(CLASS_NAMES)
x = np.arange(D_MODEL)
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = y == cls_idx
    mean_abs_delta = np.abs(delta[m]).mean(axis=0)
    ax.bar(x + cls_idx * width, mean_abs_delta, width=width,
           color=COLOURS[cls_name], label=cls_name)
ax.set_xlabel("Embedding dimension"); ax.set_ylabel("mean |h_flip - h_nom|")
ax.set_title("Per-dimension encoder-output sensitivity to hard-flip, by flavour")
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR + "embedding_dim_sensitivity.png", dpi=150, bbox_inches="tight")
print("Saved embedding_dim_sensitivity.png")

## Plot: P(b) nominal vs hard-flip

Ties the embedding-space view back to the actual tagger output.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = y == cls_idx
    sel_m = rng_plot.choice(np.where(m)[0], size=min(N_PLOT_SAMPLE, m.sum()), replace=False)
    ax.scatter(p_nom[sel_m, 0], p_flip[sel_m, 0], s=4, alpha=0.3,
               color=COLOURS[cls_name], label=cls_name, linewidths=0)
ax.axline((0, 0), slope=1, color="black", linewidth=0.8, linestyle="--", label="y = x")
ax.set_xlabel("P(b) nominal"); ax.set_ylabel("P(b) hard-flip")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.set_title("P(b): nominal vs hard-flip")
plt.tight_layout()
plt.savefig(PLOT_DIR + "pb_nom_vs_hardflip.png", dpi=150, bbox_inches="tight")
print("Saved pb_nom_vs_hardflip.png")